In [1]:
from pathlib import Path
_template = str(Path(__vsc_ipynb_file__).parent.parent / '../template.ipynb')
%run "$_template"

In [2]:
import joblib

In [3]:
df_test = pd.read_csv("../../data/modeling/test_encoded.csv")

ion = ['Ion_Chamber', 'Flux60s', 'Flux90s', 'Flux160s', 'Flux480s', 'Flux840s',
       'input_Energy', 'Temp_implantation', 'Furance_Temp', 'RTA_Temp']
X = df_test[ion].copy()
y = df_test['is_low_yield'].copy()

In [4]:
# =========================
# 1. 모델 불러오기
# =========================
ion_model = joblib.load("../../model/ion_gb.joblib")

# =========================
# 2. 불량 예측 확률 계산
# =========================
bad_prob = ion_model.predict_proba(X)[:, 1]

result = X.copy()
result["bad_prob"] = bad_prob
result["y_true"] = y.values if hasattr(y, "values") else y

# =========================
# 3. 위험구간 기준 설정
# =========================
low_threshold = 0.3
high_ratio = 0.02

high_threshold = result["bad_prob"].quantile(1 - high_ratio)

print("저위험 기준 bad_prob <=", low_threshold)
print("고위험 기준 bad_prob >=", high_threshold)

# =========================
# 4. 위험구간 부여
# =========================
def assign_risk_group(prob):
    if prob <= low_threshold:
        return "저위험"
    elif prob >= high_threshold:
        return "고위험"
    else:
        return "중위험"

result["risk_group"] = result["bad_prob"].apply(assign_risk_group)

# =========================
# 5. 위험구간별 성능 요약
# =========================
risk_summary = (
    result
    .groupby("risk_group")
    .agg(
        data_count=("y_true", "count"),
        actual_defect_count=("y_true", "sum"),
        actual_defect_rate=("y_true", "mean"),
        mean_pred_prob=("bad_prob", "mean"),
        min_pred_prob=("bad_prob", "min"),
        max_pred_prob=("bad_prob", "max")
    )
    .reset_index()
)

risk_summary["data_ratio_percent"] = risk_summary["data_count"] / len(result) * 100
risk_summary["actual_defect_rate_percent"] = risk_summary["actual_defect_rate"] * 100
risk_summary["mean_pred_prob_percent"] = risk_summary["mean_pred_prob"] * 100

display(risk_summary)

# =========================
# 6. 결과 확인
# =========================
display(result[["bad_prob", "y_true", "risk_group"]].head())

저위험 기준 bad_prob <= 0.3
고위험 기준 bad_prob >= 0.9722090573270408


,risk_group,data_count,actual_defect_count,actual_defect_rate,mean_pred_prob,min_pred_prob,max_pred_prob,data_ratio_percent,actual_defect_rate_percent,mean_pred_prob_percent
0,고위험,93,93,1.000000,0.984764,0.972334,0.996764,2.014295,100.000000,98.476435
1,저위험,4032,8,0.001984,0.012514,0.000364,0.235276,87.329435,0.198413,1.251424
2,중위험,492,483,0.981707,0.883027,0.308413,0.971943,10.656270,98.170732,88.302708


,bad_prob,y_true,risk_group
0,0.002838,0,저위험
1,0.006041,0,저위험
2,0.987534,1,고위험
3,0.003055,0,저위험
4,0.012024,0,저위험
